# Reading Difficulty Predictor — ELECTRA-large (Final Dataset)
Upload `final_train.csv`, `final_val.csv`, `final_test.csv` to `/content` before running.

**Why ELECTRA-large?**
ELECTRA is pre-trained as a discriminator (detecting replaced tokens) rather than a masked language model. This sample-efficient pre-training makes it exceptionally strong at understanding linguistic nuances. The `large` variant has 335M parameters and a hidden size of 1024, making it highly capable for complex classification tasks. Because of its size, we use a smaller batch size (8) to prevent Out-Of-Memory (OOM) errors on Colab T4.

In [ ]:
# ============================================================
#  INSTALL
# ============================================================
!pip install -q transformers scikit-learn

# ============================================================
#  IMPORTS
# ============================================================
import os, torch, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AdamW,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve
)
from sklearn.preprocessing import label_binarize
import shutil

DATA_DIR    = '/content'
SAVE_DIR    = '/content/electra_large_final'
MODEL_NAME  = 'google/electra-large-discriminator'
EPOCHS      = 5
BATCH_SIZE  = 8       # Important: Keep at 8 to avoid OOM on 16GB GPUs
MAX_LEN     = 256
LR_BERT     = 1e-5    # Lower LR for large models
LR_HEAD     = 1e-3
PATIENCE    = 2
LABEL_NAMES = ['Easy', 'Medium', 'Hard']
NUMERIC_COLS = [
    'flesch_kincaid_grade', 'word_count', 'sentence_count',
    'avg_word_length', 'avg_sentence_length', 'syllable_count',
    'unique_word_ratio', 'long_word_ratio',
    'flesch_reading_ease', 'difficult_word_percentage'
]

os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
#  LOAD DATA
# ============================================================
train_df = pd.read_csv(os.path.join(DATA_DIR, 'final_train.csv')).dropna(subset=['text','label'])
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'final_val.csv')).dropna(subset=['text','label'])
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'final_test.csv')).dropna(subset=['text','label'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Label distribution (train):\n', train_df['label'].value_counts().sort_index())

# ============================================================
#  TOKENIZER & DATASET
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {type(tokenizer).__name__}')

class ReadingDataset(Dataset):
    def __init__(self, df):
        self.texts    = df['text'].astype(str).tolist()
        self.numerics = df[NUMERIC_COLS].fillna(0).values
        self.labels   = df['label'].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'token_type_ids': enc.get('token_type_ids', torch.zeros(MAX_LEN, dtype=torch.long)).flatten(),
            'numeric_feats':  torch.tensor(self.numerics[idx], dtype=torch.float),
            'labels':         torch.tensor(self.labels[idx],   dtype=torch.long)
        }

train_loader = DataLoader(ReadingDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ReadingDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ReadingDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Verify one batch
sample = next(iter(train_loader))
print(f'Batch shapes -- input_ids: {sample["input_ids"].shape}, numeric: {sample["numeric_feats"].shape}')

# ============================================================
#  MODEL  (ELECTRA-large [CLS] + 10 numeric features -> MLP -> 3 classes)
# ============================================================
class ElectraHybrid(nn.Module):
    def __init__(self, num_labels=3, n_numeric=10):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size     = self.encoder.config.hidden_size  # 1024 for ELECTRA-large
        self.dropout    = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + n_numeric, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_labels)
        )

    def forward(self, input_ids, attention_mask, token_type_ids, numeric_feats):
        out      = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls      = self.dropout(out.last_hidden_state[:, 0, :])  # Extract [CLS] token representation
        combined = torch.cat([cls, numeric_feats], dim=1)
        return self.classifier(combined)

model = ElectraHybrid().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

# Verify forward pass
with torch.no_grad():
    test_logits = model(
        sample['input_ids'].to(device),
        sample['attention_mask'].to(device),
        sample['token_type_ids'].to(device),
        sample['numeric_feats'].to(device)
    )
print(f'Forward pass OK — logits shape: {test_logits.shape}')

# ============================================================
#  OPTIMIZER + SCHEDULER
# ============================================================
optimizer = AdamW([
    {'params': model.encoder.parameters(),    'lr': LR_BERT},
    {'params': model.classifier.parameters(), 'lr': LR_HEAD}
], weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.06)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn      = nn.CrossEntropyLoss()

# ============================================================
#  TRAINING LOOP
# ============================================================
best_val_acc     = 0.0
patience_counter = 0
CKPT             = os.path.join(SAVE_DIR, 'best_electra_large.pt')

print('\n' + '='*55)
print('        STARTING ELECTRA-LARGE TRAINING')
print('='*55)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        logits = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['token_type_ids'].to(device),
            batch['numeric_feats'].to(device)
        )
        loss = loss_fn(logits, batch['labels'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if step % 200 == 0 and step > 0:
            print(f'  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss, preds, truths = 0, [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device),
                batch['token_type_ids'].to(device),
                batch['numeric_feats'].to(device)
            )
            val_loss += loss_fn(logits, batch['labels'].to(device)).item()
            preds  += torch.argmax(logits, dim=1).cpu().tolist()
            truths += batch['labels'].tolist()

    val_acc = accuracy_score(truths, preds)
    print(f'\n--- Epoch {epoch+1}/{EPOCHS} ---')
    print(f'  Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), CKPT)
        print(f'  --> New best! Saved. (Best Val Acc: {best_val_acc:.4f})\n')
    else:
        patience_counter += 1
        print(f'  --> No improvement. Patience: {patience_counter}/{PATIENCE}\n')
        if patience_counter >= PATIENCE:
            print('  Early stopping triggered.')
            break

# ============================================================
#  FINAL TEST EVALUATION
# ============================================================
print('\nLoading best checkpoint for final evaluation...')
model.load_state_dict(torch.load(CKPT))
model.eval()

all_preds, all_truths, all_probs = [], [], []
with torch.no_grad():
    for batch in test_loader:
        logits = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['token_type_ids'].to(device),
            batch['numeric_feats'].to(device)
        )
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs  += probs.tolist()
        all_preds  += np.argmax(probs, axis=1).tolist()
        all_truths += batch['labels'].tolist()

all_probs = np.array(all_probs)

acc       = accuracy_score(all_truths, all_preds)
report    = classification_report(all_truths, all_preds, target_names=LABEL_NAMES, output_dict=True)
cm        = confusion_matrix(all_truths, all_preds)
roc_macro = roc_auc_score(all_truths, all_probs, multi_class='ovr', average='macro')
roc_wt    = roc_auc_score(all_truths, all_probs, multi_class='ovr', average='weighted')

print('\n' + '='*60)
print('   FINAL TEST RESULTS -- ELECTRA-large (FIXED)')
print('='*60)
print(f'  Accuracy          : {acc:.4f}')
print(f'  Macro  F1         : {report["macro avg"]["f1-score"]:.4f}')
print(f'  Macro  Precision  : {report["macro avg"]["precision"]:.4f}')
print(f'  Macro  Recall     : {report["macro avg"]["recall"]:.4f}')
print(f'  Weighted F1       : {report["weighted avg"]["f1-score"]:.4f}')
print(f'  Weighted Precision: {report["weighted avg"]["precision"]:.4f}')
print(f'  Weighted Recall   : {report["weighted avg"]["recall"]:.4f}')
print(f'  ROC-AUC (macro)   : {roc_macro:.4f}')
print(f'  ROC-AUC (weighted): {roc_wt:.4f}')
print()
print('Per-class:')
for name in LABEL_NAMES:
    r = report[name]
    print(f'  {name:8s} | P: {r["precision"]:.4f}  R: {r["recall"]:.4f}  F1: {r["f1-score"]:.4f}  Support: {int(r["support"])}')
print()
print('Confusion Matrix (rows=True, cols=Pred):')
print(f'         Easy  Medium  Hard')
for i, row in enumerate(cm):
    print(f'  {LABEL_NAMES[i]:8s} {row[0]:5d}  {row[1]:6d}  {row[2]:4d}')

confs   = np.max(all_probs, axis=1)
correct = np.array(all_preds) == np.array(all_truths)
print(f'\nPrediction Confidence:')
print(f'  Mean confidence (correct) : {confs[correct].mean():.4f}')
print(f'  Mean confidence (wrong)   : {confs[~correct].mean():.4f}')

# --- Save outputs ---
y_bin = label_binarize(all_truths, classes=[0, 1, 2])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, name in enumerate(LABEL_NAMES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    axes[0].plot(fpr, tpr, label=name)
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title('ROC Curves')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()
for i, name in enumerate(LABEL_NAMES):
    prec, rec, _ = precision_recall_curve(y_bin[:, i], all_probs[:, i])
    axes[1].plot(rec, prec, label=name)
axes[1].set_title('Precision-Recall Curves')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'roc_pr_curves.png'), dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels(LABEL_NAMES); ax.set_yticklabels(LABEL_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion Matrix')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150); plt.close()

test_pred_df = test_df.copy().reset_index(drop=True)
test_pred_df['predicted']   = all_preds
test_pred_df['correct']     = correct.tolist()
test_pred_df['conf_easy']   = all_probs[:, 0]
test_pred_df['conf_medium'] = all_probs[:, 1]
test_pred_df['conf_hard']   = all_probs[:, 2]
test_pred_df['confidence']  = confs
test_pred_df.to_csv(os.path.join(SAVE_DIR, 'test_predictions.csv'), index=False)
pd.DataFrame(report).T.to_csv(os.path.join(SAVE_DIR, 'classification_report.csv'))
pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES).to_csv(os.path.join(SAVE_DIR, 'confusion_matrix.csv'))

metrics = {'accuracy': acc, 'macro_f1': report['macro avg']['f1-score'],
           'weighted_f1': report['weighted avg']['f1-score'],
           'roc_auc_macro': roc_macro, 'best_val_acc': best_val_acc}
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nAll outputs saved to: {SAVE_DIR}')
shutil.make_archive('/content/electra_large_final', 'zip', SAVE_DIR)
from google.colab import files
files.download('/content/electra_large_final.zip')
print('Downloaded electra_large_final.zip')